# Portfolio optimisation: QAOA against the provable optimum

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/official-dvl/zksf/blob/main/examples/benchmarks/portfolio-optimisation.ipynb)

Reproduces the benchmark published at [zksf.org/applications/finance](https://zksf.org/applications/finance/).

Pick exactly k assets from N to minimise risk against return. The classical baseline is exhaustive enumeration, which returns the **provable** optimum rather than a heuristic estimate, so QAOA is measured against the best answer that exists.

Everything here runs on free public packages. Nothing costs anything.


In [ ]:
!pip install -q qiskit qiskit-aer scipy numpy

## The instance

A single-factor market model: every asset loads on a common factor plus idiosyncratic noise, which reproduces the correlation structure real equity covariance has. Change `SEED` or `N` and everything below re-runs on your own instance.

In [ ]:
import itertools, time
import numpy as np
from scipy.optimize import minimize
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator

SEED, N, K, LAM = 20260902, 12, 4, 1.0

def market(n, seed):
    rng = np.random.default_rng(seed + n)
    beta = rng.uniform(0.5, 1.5, n)          # factor loadings
    idio = rng.uniform(0.05, 0.25, n)        # idiosyncratic vol
    cov = 0.04 * np.outer(beta, beta) + np.diag(idio ** 2)
    mu = rng.uniform(0.02, 0.15, n)          # expected excess returns
    return mu, cov

mu, cov = market(N, SEED)
obj = lambda x: float(x @ cov @ x - LAM * (mu @ x))
print(f"{N} assets, choose {K}, {len(list(itertools.combinations(range(N), K))):,} portfolios")

## Classical baseline: the provable optimum

Enumeration also gives the full distribution of outcomes, which is what lets us score QAOA on a scale that stays meaningful when the optimum sits near zero. Percentage gap does not: at N=14 the optimum is 0.000495, where a negligible miss reads as a 28,000 percent error.

In [ ]:
t0 = time.perf_counter()
spectrum = np.sort([obj(np.eye(N)[list(c)].sum(0)) for c in itertools.combinations(range(N), K)])
t_classical = time.perf_counter() - t0
best, worst = float(spectrum[0]), float(spectrum[-1])
print(f"optimum {best:.6f}   worst {worst:.6f}   in {t_classical*1000:.1f} ms")

## QAOA

The cardinality constraint becomes a quadratic penalty. Note the evaluation counter: a variational algorithm runs the circuit hundreds of times inside the optimiser, and that is the real cost. A benchmark reporting only the final circuit understates it by two orders of magnitude.

In [ ]:
pen = float(np.max(np.abs(cov)) * N + LAM * np.max(mu)) * 2.0
Q = cov + pen * (np.ones((N, N)) - np.eye(N))
np.fill_diagonal(Q, np.diag(Q) - LAM * mu + pen * (1 - 2 * K) + pen)

sim, evals = AerSimulator(), {"n": 0}

def qaoa(gammas, betas):
    Qs = (Q + Q.T) / 2
    qc = QuantumCircuit(N); qc.h(range(N))
    J = {(i, j): Qs[i, j] / 4 for i in range(N) for j in range(i + 1, N)}
    h = [-Qs[i, i] / 2 - sum(Qs[i, j] for j in range(N) if j != i) / 4 for i in range(N)]
    for g, b in zip(gammas, betas):
        for (i, j), v in J.items():
            if abs(v) > 1e-12: qc.rzz(2 * g * v, i, j)
        for i in range(N):
            if abs(h[i]) > 1e-12: qc.rz(2 * g * h[i], i)
        for i in range(N): qc.rx(2 * b, i)
    qc.measure_all(); return qc

def best_sample(params, p):
    evals["n"] += 1
    counts = sim.run(qaoa(params[:p], params[p:]), shots=512).result().get_counts()
    out = np.inf
    for bits in counts:
        x = np.array([int(c) for c in reversed(bits)], dtype=float)
        if int(x.sum()) == K: out = min(out, obj(x))   # feasible only
    return out

## Run it

Three restarts at 150 iterations. QAOA at higher depth contains lower depth as a special case, so if a deeper run scores worse the optimiser failed to converge, not the algorithm. Under-optimising the quantum side would be a strawman.

In [ ]:
rng = np.random.default_rng(SEED)
for p in (1, 2, 3):
    t0 = time.perf_counter(); evals["n"] = 0; found = np.inf
    for r in range(3):
        x0 = np.r_[np.full(p, 0.6), np.full(p, 0.4)] if r == 0 else rng.uniform(0, np.pi, 2 * p)
        res = minimize(lambda t: min(1e6, best_sample(t, p)), x0,
                       method="COBYLA", options={"maxiter": 150, "rhobeg": 0.4})
        found = min(found, best_sample(res.x, p))
    dt = time.perf_counter() - t0
    norm = (found - best) / (worst - best)
    beats = (spectrum > found).sum() / len(spectrum) * 100
    print(f"p={p}  QAOA {found:.6f}  norm {norm:.4f}  beats {beats:.1f}%  "
          f"{dt:.1f}s  {evals['n']} evals  optimal={abs(found-best)<1e-12}")

## What you should see

QAOA lands in the top couple of percent of all feasible portfolios and sometimes hits the exact optimum. It also takes seconds against milliseconds, and **it cannot prove anything**. Enumeration returns the optimum and a proof that nothing better exists; QAOA returns a good portfolio and a shrug.

The published run found the optimum in 6 of 12 configurations. Yours will differ slightly because Aer's sampling is not seeded here.

## The caveat that matters

Enumeration is not what a desk uses. Production practice is mixed-integer quadratic programming with branch and bound, which handles universes in the hundreds. We use enumeration because it *proves* optimality at sizes a quantum engine can also run. Any claim that quantum is close has to beat Gurobi at N in the hundreds, not enumeration at N=50.


## Why your numbers will not match exactly

Two reasons, both worth understanding before you compare against the published table.

**Sampling is not seeded.** QAOA reads its answer from measurement counts, so every run explores slightly differently and the optimiser lands somewhere slightly different. Expect the same *shape*, not the same digits: a result in the top couple of percent of feasible solutions, hitting the exact optimum sometimes and not others. If you re-run this cell a few times you will see that spread directly, and that spread is itself the honest finding about QAOA's reliability.

**The published timings came through the ZKSF engine**, which adds job handling, routing and certification around the same simulation. This notebook calls Aer directly, so it is faster here. The algorithm and the answer quality are the same; only the wall clock differs.

The classical numbers do reproduce exactly wherever they are proven optimal, because a proof is not a sample.


## Next

- [The full benchmark page](https://zksf.org/applications/finance/), with the analysis and the caveats
- [How we benchmark](https://zksf.org/applications/methodology/): the rules every one of these follows
- [All applications](https://zksf.org/applications/) across six sectors
- [Certification](https://zksf.org/quantum-computing-certification/): what the accuracy statements assert
